# Домашнє завдання 5 — Версіонування подій та Idempotent Consumer

Реалізація двох ключових патернів EDA:
1. **Версіонування** — evolving events across versions with upcasting
2. **Ідемпотентний обробник** — deduplication table ensures at-most-once business logic

## 1. Версіонування подій

In [ ]:
from __future__ import annotations

import json
import logging
import sqlite3
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from decimal import Decimal
from uuid import UUID, uuid4

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s  %(message)s",
)
log = logging.getLogger(__name__)

print("Imports OK")

In [ ]:
# --- Event v1 (original) ---

@dataclass(frozen=True)
class OrderPlacedV1:
    """Original event schema — stored in the event store for older orders."""

    event_id: UUID
    order_id: UUID
    customer_id: str
    total_amount: Decimal
    delivery_address: str   # will be renamed in v2
    version: int = 1


# --- Event v2 (new) ---

@dataclass(frozen=True)
class OrderPlacedV2:
    """New event schema — shipping_address rename + loyalty_tier addition."""

    event_id: UUID
    order_id: UUID
    customer_id: str
    total_amount: Decimal
    shipping_address: str   # перейменовано з delivery_address
    loyalty_tier: str       # нове поле
    version: int = 2


print("OrderPlacedV1 and OrderPlacedV2 defined")

### 1.1 Upcaster v1 → v2

Upcaster перетворює стару подію у нову без зміни сховища подій.
Відбувається **під час читання** — event store залишається незмінним.

In [ ]:
def upcast_v1_to_v2(v1: OrderPlacedV1) -> OrderPlacedV2:
    """Map v1 fields onto v2 schema, supplying defaults for new fields."""
    return OrderPlacedV2(
        event_id=v1.event_id,
        order_id=v1.order_id,
        customer_id=v1.customer_id,
        total_amount=v1.total_amount,
        shipping_address=v1.delivery_address,  # rename
        loyalty_tier="Standard",                # default for pre-tier orders
    )


# --- Demo ---

v1_event = OrderPlacedV1(
    event_id=uuid4(),
    order_id=uuid4(),
    customer_id="cust-42",
    total_amount=Decimal("350.00"),
    delivery_address="вул. Хрещатик, 1, Київ",
)

v2_event = upcast_v1_to_v2(v1_event)

assert v2_event.shipping_address == v1_event.delivery_address
assert v2_event.loyalty_tier == "Standard"
assert v2_event.order_id == v1_event.order_id
assert v2_event.version == 2

print(f"v1 delivery_address : {v1_event.delivery_address!r}")
print(f"v2 shipping_address : {v2_event.shipping_address!r}")
print(f"v2 loyalty_tier     : {v2_event.loyalty_tier!r}")
print("Upcaster assertions passed")

### 1.2 Десеріалізація з автоматичним upcasting

Deserializer перевіряє поле `version` у JSON і автоматично upcast-ить v1 у v2,
щоб вся решта системи завжди отримувала `OrderPlacedV2`.

In [ ]:
def deserialize_order_placed(raw_json: str) -> OrderPlacedV2:
    """Parse raw JSON into OrderPlacedV2, upcasting v1 if necessary."""
    data = json.loads(raw_json)
    version = data.get("version", 1)

    match version:
        case 1:
            v1 = OrderPlacedV1(
                event_id=UUID(data["event_id"]),
                order_id=UUID(data["order_id"]),
                customer_id=data["customer_id"],
                total_amount=Decimal(str(data["total_amount"])),
                delivery_address=data["delivery_address"],
            )
            return upcast_v1_to_v2(v1)
        case 2:
            return OrderPlacedV2(
                event_id=UUID(data["event_id"]),
                order_id=UUID(data["order_id"]),
                customer_id=data["customer_id"],
                total_amount=Decimal(str(data["total_amount"])),
                shipping_address=data["shipping_address"],
                loyalty_tier=data["loyalty_tier"],
            )
        case _:
            raise ValueError(f"Unsupported event version: {version}")


# --- Demo: v1 JSON payload (як якби прийшло зі старого сервісу) ---

raw_v1 = json.dumps({
    "version": 1,
    "event_id": str(uuid4()),
    "order_id": str(uuid4()),
    "customer_id": "cust-99",
    "total_amount": "120.50",
    "delivery_address": "пр. Перемоги, 10, Київ",
})

result = deserialize_order_placed(raw_v1)
assert isinstance(result, OrderPlacedV2)
assert result.loyalty_tier == "Standard"
print(f"Deserialized v1 JSON → {result.__class__.__name__}")
print(f"  shipping_address : {result.shipping_address!r}")
print(f"  loyalty_tier     : {result.loyalty_tier!r}")

# --- Demo: v2 JSON payload ---

raw_v2 = json.dumps({
    "version": 2,
    "event_id": str(uuid4()),
    "order_id": str(uuid4()),
    "customer_id": "cust-77",
    "total_amount": "890.00",
    "shipping_address": "вул. Велика Васильківська, 55, Київ",
    "loyalty_tier": "Gold",
})

result_v2 = deserialize_order_placed(raw_v2)
assert result_v2.loyalty_tier == "Gold"
print(f"Deserialized v2 JSON → {result_v2.__class__.__name__}")
print(f"  shipping_address : {result_v2.shipping_address!r}")
print(f"  loyalty_tier     : {result_v2.loyalty_tier!r}")

### 1.3 Safe зміни vs Breaking зміни

#### Safe (non-breaking) зміна — додавання нового необов'язкового поля

**Приклад:** `LoyaltyTier` у `OrderPlacedV2`.

- Старі consumers, що не знають про це поле, просто **ігнорують** його — ніякого збою.
- Нові consumers отримують значення. Якщо читається стара подія (v1) — upcaster підставляє **дефолт** (`"Standard"`).
- Event store залишається незмінним.
- **Правило:** нові поля завжди мають дефолт або є nullable.

#### Breaking зміна — перейменування поля

**Приклад:** `delivery_address` → `shipping_address`.

- Старі consumers шукають `delivery_address` — **падають** з KeyError або отримують `None`.
- Нові consumers шукають `shipping_address` — **падають** на старих подіях.
- Варіанти вирішення:
  1. **Upcaster (обрано тут):** під час читання v1 → v2, перейменовуємо «на льоту».
  2. **Dual-write period:** тимчасово писати обидва поля (`delivery_address` + `shipping_address`) до повної міграції consumers.
  3. **In-place migration:** переписати всі події в сховищі (ризиковано, потребує downtime).

> **Висновок:** будь-яке видалення або перейменування поля є breaking change.
> Додавання поля з дефолтом — safe, якщо upcaster заповнює його для старих версій.

## 2. Ідемпотентний обробник (Idempotent Consumer)

### 2.1 Схема БД

Використовуємо `sqlite3` (stdlib) з in-memory базою для демонстрації.
У production це була б таблиця в PostgreSQL / SQL Server.

```sql
-- Таблиця дедуплікації
CREATE TABLE processed_messages (
    message_id   TEXT NOT NULL,
    handler_name TEXT NOT NULL,
    processed_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (message_id, handler_name)
);

-- Бізнес-таблиця (бонусні бали)
CREATE TABLE loyalty_points (
    customer_id  TEXT PRIMARY KEY,
    total_points INTEGER NOT NULL DEFAULT 0,
    last_updated TEXT    NOT NULL
);
```

In [ ]:
def create_db() -> sqlite3.Connection:
    """Create an in-memory SQLite database with the dedup and loyalty tables."""
    conn = sqlite3.connect(":memory:")
    conn.execute("""
        CREATE TABLE processed_messages (
            message_id   TEXT NOT NULL,
            handler_name TEXT NOT NULL,
            processed_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (message_id, handler_name)
        )
    """)
    conn.execute("""
        CREATE TABLE loyalty_points (
            customer_id  TEXT PRIMARY KEY,
            total_points INTEGER NOT NULL DEFAULT 0,
            last_updated TEXT    NOT NULL
        )
    """)
    conn.commit()
    return conn

print("DB schema defined")

### 2.2 OrderPlacedHandler з дедуплікацією

Патерн: **перевірка → бізнес-логіка → запис в dedup — все в одній транзакції**.

```
BEGIN
  1. SELECT processed_messages WHERE message_id = ?  → вже оброблено? → COMMIT + return
  2. Нарахувати бонусні бали (UPSERT loyalty_points)
  3. INSERT INTO processed_messages
COMMIT
```

Якщо між кроком 2 і 3 стався збій — транзакція відкочується,
повідомлення повертається в чергу і буде оброблено знову (at-least-once delivery).
Завдяки dedup table повторна обробка є ідемпотентною.

In [ ]:
@dataclass(frozen=True)
class OrderPlacedMessage:
    """Message contract consumed from the queue."""

    message_id: UUID
    order_id: UUID
    customer_id: str
    total_amount: Decimal


HANDLER_NAME = "OrderPlacedHandler"


class OrderPlacedHandler:
    """Idempotent consumer: deduplication via processed_messages table."""

    def __init__(self, conn: sqlite3.Connection) -> None:
        self._conn = conn

    def handle(self, message: OrderPlacedMessage) -> None:
        with self._conn:
            # Step 1: check dedup table
            row = self._conn.execute(
                "SELECT 1 FROM processed_messages "
                "WHERE message_id = ? AND handler_name = ?",
                (str(message.message_id), HANDLER_NAME),
            ).fetchone()

            if row is not None:
                log.warning(
                    "Duplicate message %s — skipping",
                    message.message_id,
                )
                return  # ACK: message already processed

            # Step 2: business logic — accrue loyalty points
            points = self._calculate_points(message.total_amount)
            now = datetime.now(timezone.utc).isoformat()

            self._conn.execute(
                """
                INSERT INTO loyalty_points (customer_id, total_points, last_updated)
                VALUES (?, ?, ?)
                ON CONFLICT(customer_id) DO UPDATE SET
                    total_points = total_points + excluded.total_points,
                    last_updated = excluded.last_updated
                """,
                (message.customer_id, points, now),
            )

            # Step 3: record in dedup table
            self._conn.execute(
                "INSERT INTO processed_messages (message_id, handler_name) VALUES (?, ?)",
                (str(message.message_id), HANDLER_NAME),
            )

            log.info(
                "Processed order %s: +%d points for %s",
                message.order_id,
                points,
                message.customer_id,
            )

    @staticmethod
    def _calculate_points(amount: Decimal) -> int:
        """1 point per every 10 UAH."""
        return int(amount // 10)


print("OrderPlacedHandler defined")

### 2.3 Тест: 3 рази відправити → бізнес-операція виконається 1 раз

In [ ]:
# Setup
db = create_db()
handler = OrderPlacedHandler(db)

message = OrderPlacedMessage(
    message_id=uuid4(),
    order_id=uuid4(),
    customer_id="cust-42",
    total_amount=Decimal("350.00"),  # → 35 points
)

print("--- Sending message 3 times ---")
for i in range(1, 4):
    print(f"\nSend #{i}")
    handler.handle(message)

# --- Assertions ---

# Only 1 row in processed_messages
dedup_count = db.execute(
    "SELECT COUNT(*) FROM processed_messages WHERE message_id = ?",
    (str(message.message_id),),
).fetchone()[0]
assert dedup_count == 1, f"Expected 1 dedup row, got {dedup_count}"

# Business logic ran exactly once: points = 350 // 10 = 35
points = db.execute(
    "SELECT total_points FROM loyalty_points WHERE customer_id = ?",
    (message.customer_id,),
).fetchone()[0]
assert points == 35, f"Expected 35 points, got {points}"

print(f"\n--- Results ---")
print(f"processed_messages rows : {dedup_count}  (expected 1)")
print(f"loyalty_points accrued  : {points}  (expected 35)")
print("\nAll assertions passed — бізнес-операція виконалась рівно 1 раз")

## Архітектура

### Частина 1 — Версіонування

```
Event Store
  └─ OrderPlacedV1 (старі події)
  └─ OrderPlacedV2 (нові події)
              ↓
        deserialize_order_placed(raw_json)
              ↓ version == 1
        upcast_v1_to_v2()
              ↓
        OrderPlacedV2  ← єдиний тип, що бачить решта системи
```

### Частина 2 — Idempotent Consumer

```
Queue (RabbitMQ / Azure Service Bus)
  └─ OrderPlacedMessage (може прийти 1+ разів)
              ↓
        BEGIN TRANSACTION
          1. SELECT processed_messages → duplicate? → COMMIT + return
          2. UPSERT loyalty_points (+points)
          3. INSERT processed_messages
        COMMIT
              ↓
        ACK message
```

Транзакційність гарантує: або **всі три кроки** виконались разом, або жоден.
Повторна доставка повідомлення безпечна — деdup table зупиняє повторне нарахування балів.